# Sun-Synchronous Orbit — J2 Precession Study

This notebook analyses the 1-year trajectory of a single satellite at 600 km, 97.78°
inclination simulated with full J2 + drag + SRP force models.

**Theory**: J2 causes a secular drift in the Right Ascension of the Ascending Node (RAAN):

$$\dot{\Omega}_{J2} = -\frac{3}{2} \, n \, J_2 \left(\frac{R_E}{a}\right)^2 \cos i$$

For a Sun-synchronous orbit we require $\dot{\Omega} = +0.9856°/\text{day}$
(Earth's mean solar motion), which forces a retrograde inclination $i \approx 97.78°$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.grid': True,
    'grid.alpha': 0.35,
    'lines.linewidth': 1.4,
})

In [ ]:
# ---------------------------------------------------------------------------
# Load trajectory data
# ---------------------------------------------------------------------------
import pathlib

# Look for the most recent sso_j2_study run
base = pathlib.Path('../build/output/sso_j2_study/run_0000')
df = pd.read_csv(base / 'trajectory.csv')

df['time_days'] = df['time_s'] / 86400.0

print(f'Rows: {len(df):,}   Columns: {list(df.columns)}')
df.head(3)

## 1 · RAAN Precession

The raw RAAN is wrapped to $[0°, 360°)$.  
We unwrap it so the linear trend is visible, then fit a line to extract the measured precession rate.

In [ ]:
# Unwrap RAAN: convert degrees to radians, unwrap, convert back
raan_rad    = np.deg2rad(df['raan_deg'].values)
raan_unwrap = np.rad2deg(np.unwrap(raan_rad))

# Linear regression to extract the precession rate
slope, intercept, r, p, se = stats.linregress(df['time_days'], raan_unwrap)
print(f'Measured precession rate : {slope:.5f} deg/day')
print(f'Ideal SSO rate           : +0.98565 deg/day')
print(f'Relative error           : {abs(slope - 0.98565)/0.98565*100:.3f} %')
print(f'R²                       : {r**2:.7f}')

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df['time_days'], raan_unwrap,  color='steelblue',  label='Simulated RAAN (unwrapped)')
ax.plot(df['time_days'],
        intercept + slope * df['time_days'],
        'r--', lw=1.8, label=f'Linear fit  {slope:+.5f}°/day')
ax.axhline(0,  color='gray', lw=0.6)
ax.set_xlabel('Elapsed time (days)')
ax.set_ylabel('RAAN (deg)')
ax.set_title('J2-driven RAAN Precession of the SSO (600 km, 97.78°)')
ax.legend()
ax.yaxis.set_major_locator(mticker.MultipleLocator(60))
plt.tight_layout()
plt.show()

## 2 · Inclination Stability

J2 secular precession does **not** change the inclination — only short-periodic oscillations
should be visible.  A long-term drift here would indicate a model error.

In [ ]:
inc_mean   = df['inclination_deg'].mean()
inc_std    = df['inclination_deg'].std()
inc_range  = df['inclination_deg'].max() - df['inclination_deg'].min()

print(f'Mean inclination : {inc_mean:.5f}°')
print(f'Std deviation    : {inc_std:.6f}°')
print(f'Peak-to-peak     : {inc_range:.6f}°')

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(df['time_days'], df['inclination_deg'],
        color='darkorange', alpha=0.75, lw=0.8)
ax.axhline(inc_mean,    color='red',  lw=1.5, ls='--', label=f'Mean = {inc_mean:.4f}°')
ax.axhline(97.78,       color='lime', lw=1.2, ls=':',  label='Configured 97.78°')
ax.set_xlabel('Elapsed time (days)')
ax.set_ylabel('Inclination (deg)')
ax.set_title('Inclination Stability over 1 Year')
ax.legend()
plt.tight_layout()
plt.show()

## 3 · Orbital Altitude Decay (Drag)

At 600 km drag is non-negligible.  We plot both the raw instantaneous altitude and
a rolling median to show the secular decay beneath the oscillating eccentricity.

In [ ]:
window = 24  # 24-hour rolling median
alt_median = df['altitude_km'].rolling(window=window, center=True).median()

decay_km = df['altitude_km'].iloc[0] - alt_median.dropna().iloc[-1]
decay_per_day = decay_km / df['time_days'].iloc[-1]
print(f'Total altitude decay      : {decay_km:.2f} km over {df["time_days"].iloc[-1]:.1f} days')
print(f'Mean decay rate           : {decay_per_day*1000:.2f} m/day')

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df['time_days'], df['altitude_km'],
        color='mediumpurple', alpha=0.25, lw=0.5, label='Instantaneous altitude')
ax.plot(df['time_days'], alt_median,
        color='purple', lw=1.8, label=f'{window}-hr rolling median')
ax.set_xlabel('Elapsed time (days)')
ax.set_ylabel('Altitude (km)')
ax.set_title('Altitude Decay from Atmospheric Drag over 1 Year')
ax.legend()
plt.tight_layout()
plt.show()

## 4 · Analytical Verification: J2 Precession Rate vs Altitude

The secular RAAN drift for a circular orbit is:

$$\dot{\Omega} = -\frac{3}{2} \, n \, J_2 \left(\frac{R_E}{a}\right)^2 \cos i
   \qquad [\text{rad/s}]$$

For the SSO inclination, $\cos i < 0$, making $\dot{\Omega} > 0$ (eastward precession).
We sweep altitude from 400 – 1200 km and show both the precession rate and the
required SSO inclination.

In [ ]:
# Constants
MU   = 3.986004418e14   # m^3/s^2
RE   = 6.3781366e6      # m  (WGS-84)
J2   = 1.08262668e-3
SUN_RATE_RAD_S = 0.985647 * np.pi / 180.0 / 86400.0  # rad/s

alt_km  = np.linspace(300, 1200, 500)
a_m     = RE + alt_km * 1e3
n       = np.sqrt(MU / a_m**3)

# SSO inclination: cos(i) = -SUN_RATE / (1.5 * n * J2 * (RE/a)^2)
cos_i_sso = -SUN_RATE_RAD_S / (1.5 * n * J2 * (RE / a_m)**2)
# Clamp for safety (very low altitudes may push cos out of [-1,1])
cos_i_sso = np.clip(cos_i_sso, -1.0, 1.0)
inc_sso_deg = np.rad2deg(np.arccos(cos_i_sso))

# RAAN precession rate at the SSO inclination (should be ~+0.9856 deg/day everywhere)
prec_rate_deg_day = np.rad2deg(
    -1.5 * n * J2 * (RE / a_m)**2 * cos_i_sso) * 86400

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(alt_km, inc_sso_deg, color='steelblue')
ax1.axvline(600, color='red', ls='--', lw=1.2, label='This study (600 km)')
ax1.axhline(97.78, color='orange', ls=':', lw=1.2, label='97.78°')
ax1.set_xlabel('Altitude (km)')
ax1.set_ylabel('SSO Inclination (deg)')
ax1.set_title('Required Inclination for SSO vs Altitude')
ax1.legend(fontsize=9)

ax2.plot(alt_km, prec_rate_deg_day, color='darkorange')
ax2.axhline(0.985647, color='red', ls='--', lw=1.2, label='Target 0.9856°/day')
ax2.axvline(600, color='steelblue', ls=':', lw=1.2)
ax2.set_xlabel('Altitude (km)')
ax2.set_ylabel('RAAN precession rate (deg/day)')
ax2.set_title('J2 Precession Rate at SSO Inclination vs Altitude')
ax2.set_ylim(0.95, 1.02)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 5 · Gravity Force vs Latitude (J2 Effect)

The oblate Earth ($J_2 \neq 0$) means gravity is slightly **stronger at the poles** than
at the equator.  The radial gravitational acceleration on the surface at geocentric
latitude $\phi$ is:

$$g(\phi) \approx \frac{\mu}{R_E^2}
  \left[1 - \frac{3}{2}J_2 \left(3\sin^2\phi - 1\right)\right]$$

The full WGS-84 formula (Somigliani, 1929) gives the 'normal gravity' accounting for the
actual ellipsoidal shape and Earth's rotation.  We plot both for comparison.

In [ ]:
lat_deg = np.linspace(-90, 90, 500)
lat_rad = np.deg2rad(lat_deg)

# --- J2-only point-mass approximation (radial component) ---
g_j2 = (MU / RE**2) * (1.0 - 1.5 * J2 * (3 * np.sin(lat_rad)**2 - 1))

# --- WGS-84 Somigliani normal gravity ---
# γ(φ) = γ_e (1 + k sin²φ) / sqrt(1 - e² sin²φ)
gamma_e = 9.7803267715     # m/s² equatorial normal gravity
gamma_p = 9.8321863685     # m/s² polar normal gravity
e2      = 0.00669437999014  # first eccentricity squared (WGS-84)
k       = (RE / 6356752.3142) * (gamma_p / gamma_e) - 1  # ~0.00193
g_wgs84 = gamma_e * (1.0 + k * np.sin(lat_rad)**2) / np.sqrt(1 - e2 * np.sin(lat_rad)**2)

# --- Percentage effect of J2 relative to Newtonian point-mass ---
g_point = MU / RE**2  # ~9.8197 m/s²
pct_j2  = (g_j2 - g_point) / g_point * 100.0

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel 1: Absolute gravity
axes[0].plot(lat_deg, g_j2,   color='steelblue',  label='J2 approximation')
axes[0].plot(lat_deg, g_wgs84, color='darkorange', ls='--', lw=1.2, label='WGS-84 normal gravity')
axes[0].axhline(g_point, color='gray', ls=':', lw=1.0, label='Point-mass μ/R²')
axes[0].set_xlabel('Latitude (deg)')
axes[0].set_ylabel('g  (m/s²)')
axes[0].set_title('Surface Gravity vs Latitude')
axes[0].legend(fontsize=8)
axes[0].xaxis.set_major_locator(mticker.MultipleLocator(30))

# Panel 2: J2 correction only (milli-g units)
dg_mg = (g_j2 - g_point) * 1e3   # milli-m/s²  (close enough to milli-g)
axes[1].plot(lat_deg, dg_mg, color='firebrick')
axes[1].axhline(0, color='gray', lw=0.7)
axes[1].set_xlabel('Latitude (deg)')
axes[1].set_ylabel('Δg  (mm/s²)')
axes[1].set_title('J2 Gravity Correction vs Latitude')
axes[1].xaxis.set_major_locator(mticker.MultipleLocator(30))

# Panel 3: J2 perturbation acceleration felt by an orbiting satellite at 600 km
a_study_m  = RE + 600e3
r          = a_study_m
# J2 radial perturbation at orbit altitude
da_r  = -(MU * RE**2 * J2 / r**4) * 1.5 * (3 * np.sin(lat_rad)**2 - 1)
# J2 latitudinal (north–south) perturbation
da_phi = -(MU * RE**2 * J2 / r**4) * 3.0 * np.sin(lat_rad) * np.cos(lat_rad)
da_total = np.sqrt(da_r**2 + da_phi**2) * 1e6  # convert to μm/s²

axes[2].plot(lat_deg, da_r   * 1e6, color='steelblue',  label='Radial Δa_r',      lw=1.3)
axes[2].plot(lat_deg, da_phi * 1e6, color='darkorange',  label='Latitudinal Δa_φ', lw=1.3)
axes[2].plot(lat_deg, da_total,     color='black',        label='|Total|',          lw=1.5, ls='--')
axes[2].axhline(0, color='gray', lw=0.7)
axes[2].set_xlabel('Orbital latitude (deg)')
axes[2].set_ylabel('J2 perturbation (μm/s²)')
axes[2].set_title('J2 Perturbation Acceleration at 600 km')
axes[2].legend(fontsize=8)
axes[2].xaxis.set_major_locator(mticker.MultipleLocator(30))

plt.tight_layout()
plt.show()

print(f'Point-mass surface gravity μ/R²  : {g_point:.5f} m/s²')
print(f'J2 g at equator                  : {g_j2[250]:.5f} m/s²')
print(f'J2 g at pole                     : {g_j2[-1]:.5f} m/s²')
print(f'Pole–equator difference          : {(g_j2[-1]-g_j2[250])*1000:.2f} mm/s²')
print(f'WGS-84 g at equator              : {g_wgs84[250]:.5f} m/s²')
print(f'WGS-84 g at pole                 : {g_wgs84[-1]:.5f} m/s²')

## 6 · Measured vs Analytical Precession Rate Comparison

In [ ]:
# Analytical prediction at the mean simulated SMA and inclination
a_mean_m   = df['sma_km'].mean() * 1e3
inc_mean_r = np.deg2rad(df['inclination_deg'].mean())
n_mean     = np.sqrt(MU / a_mean_m**3)

prec_analytical = np.rad2deg(
    -1.5 * n_mean * J2 * (RE / a_mean_m)**2 * np.cos(inc_mean_r)
) * 86400  # deg/day

print('='*55)
print(f'Mean SMA over simulation   : {a_mean_m/1e3:.3f} km')
print(f'Mean inclination           : {np.rad2deg(inc_mean_r):.5f}°')
print(f'Analytical precession rate : {prec_analytical:.5f} deg/day')
print(f'Simulated precession rate  : {slope:.5f} deg/day')
print(f'Target SSO rate            : 0.98565 deg/day')
print(f'Error (sim vs analytical)  : {abs(slope - prec_analytical)/prec_analytical*100:.4f}%')
print('='*55)